In [73]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

import time
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm import tqdm
import copy
import pandas as pd




Parameters

In [74]:
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
NUM_EPOCHS = 50
NUM_WORKERS = 2
IMAGE_HEIGHT = 90  
IMAGE_WIDTH = 90   
PIN_MEMORY = True
LOAD_MODEL = False
TRAIN_DIR = "/home/st-juho/code_testing/dataset_downscaled_pngs/train"
VAL_DIR = "/home/st-juho/code_testing/dataset_downscaled_pngs/val"

Code

U-Net definitions

In [75]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)
    
class UNET(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNET, self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Down part of UNET
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Up part of UNET
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(
                    feature*2, feature, kernel_size=2, stride=2,
                )
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            #Check if sizes are same, change method if needed / wanted
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.final_conv(x)
    

def test():
    x = torch.randn((3,1,160,160))
    model = UNET(in_channels=1, out_channels=1)
    preds = model(x)
    print(preds.shape)
    print(x.shape)
    assert preds.shape == x.shape

if __name__ == "__main__":
    test()

torch.Size([3, 1, 160, 160])
torch.Size([3, 1, 160, 160])


Importing functions

In [76]:
class InSARDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Find all mask tiles
        self.mask_files = sorted([f for f in os.listdir(root_dir) if f.endswith("_mask.png")])

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, index):
        mask_filename = self.mask_files[index]
        
        # Logic to find matching Amp/Coh files:
        # Mask: flood_01_y0_x0_mask.png
        # Amp:  flood_01_y0_x0_amp.png
        # Coh:  flood_01_y0_x0_coh.png
        
        amp_filename = mask_filename.replace("_mask.png", "_amp.png")
        coh_filename = mask_filename.replace("_mask.png", "_coh.png")

        amp_path = os.path.join(self.root_dir, amp_filename)
        coh_path = os.path.join(self.root_dir, coh_filename)
        mask_path = os.path.join(self.root_dir, mask_filename)

        # Load
        amp = np.array(Image.open(amp_path).convert("L"), dtype=np.float32)
        coh = np.array(Image.open(coh_path).convert("L"), dtype=np.float32)
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)

        # Stack
        image = np.stack([amp, coh], axis=-1)

        # Normalize Mask
        mask[mask == 255.0] = 1.0
        mask[mask > 1.0] = 1.0

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations['image']
            mask = augmentations['mask']

        return image, mask

Utility functions

In [77]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])

def get_loaders(
    train_dir,
    train_maskdir,
    val_dir,
    val_maskdir,
    batch_size,
    train_transform,
    val_transform,
    num_workers=4,
    pin_memory=True,
):
    train_ds = InSARDataset(
        root_dir=TRAIN_DIR,
        transform=train_transform,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_ds = InSARDataset(
        root_dir=VAL_DIR,
        transform=val_transform,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).unsqueeze(1)

            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)

    print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}%")
    print(f"Dice score: {dice_score/len(loader)}")
    model.train()

def save_predictions_as_imgs(loader, model, folder="saved_images/", device="cuda"):
    model.eval()
    if not os.path.exists(folder):
        os.makedirs(folder)
        
    for idx, (x, y) in enumerate(loader):
        x = x.to(device=device)
        with torch.no_grad():
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
        
        # Save Prediction
        torchvision.utils.save_image(
            preds, f"{folder}/pred_{idx}.png"
        )
        # Save Ground Truth
        torchvision.utils.save_image(y.float().unsqueeze(1), f"{folder}/true_{idx}.png")
        
        # OPTIONAL: Save Inputs (Amplitude only, for reference)
        # x is [Batch, 2, H, W]. Slice 0 is Amplitude.
        torchvision.utils.save_image(x[:, 0:1, :, :], f"{folder}/input_amp_{idx}.png")

    model.train()

In [78]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().to(device=DEVICE).unsqueeze(1)

        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # update tqdm loop
        loop.set_postfix(loss=loss.item())

def main():
    # --- CONFIGURATION ---
    # Point these to your new folders containing the _amp.png, _coh.png, _mask.png files
    # Make sure you have physically moved the files into 'train' and 'val' subfolders!
    TRAIN_DIR = "/home/st-juho/code_testing/dataset_downscaled_pngs/train"
    VAL_DIR = "/home/st-juho/code_testing/dataset_downscaled_pngs/val"
    
    # --- TRANSFORMS ---
    # Critical: Use RandomCrop because your images are 9000x9000
    train_transform = A.Compose(
        [
            A.Rotate(limit=35, p=1.0),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.1),
            A.Normalize(
                mean=[0.0, 0.0],  # 2 Channels
                std=[1.0, 1.0],   # 2 Channels
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )

    val_transform = A.Compose(
        [
            A.Normalize(
                mean=[0.0, 0.0],
                std=[1.0, 1.0],
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )

    # --- DATASET & LOADERS ---
    # 1. Create Train Dataset
    train_ds = InSARDataset(
        root_dir=TRAIN_DIR,
        transform=train_transform,
    )
    
    # 2. Create Validation Dataset (This was missing in your error!)
    val_ds = InSARDataset(
        root_dir=VAL_DIR,
        transform=val_transform,
    )

    # 3. Create Loaders
    train_loader = DataLoader(
        train_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS, 
        pin_memory=PIN_MEMORY
    )

    # --- MODEL SETUP ---
    # Change in_channels=2 (Amp + Coh)
    model = UNET(in_channels=2, out_channels=1).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    if LOAD_MODEL:
        load_checkpoint(torch.load("my_checkpoint.pth.tar"), model)

    scaler = torch.cuda.amp.GradScaler()

    # --- TRAINING LOOP ---
    for epoch in range(NUM_EPOCHS):
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
        save_checkpoint(checkpoint)
        
        check_accuracy(val_loader, model, device=DEVICE)
        
        # Save example images
        save_predictions_as_imgs(
            val_loader, model, folder="saved_images/", device=DEVICE
        )

if __name__ == "__main__":
    main()

/tmp/ipykernel_1257403/4165805715.py:95: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch [1/50]


  0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_1257403/4165805715.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 1/1 [00:00<00:00,  3.87it/s, loss=0.828]


=> Saving checkpoint
Got 6086/8100 with acc 75.14%
Dice score: 0.0
Epoch [2/50]


100%|██████████| 1/1 [00:00<00:00,  4.41it/s, loss=0.805]


...

=> Saving checkpoint
Got 6086/8100 with acc 75.14%
Dice score: 0.0
Epoch [49/50]


100%|██████████| 1/1 [00:00<00:00,  3.87it/s, loss=0.442]


=> Saving checkpoint
Got 6086/8100 with acc 75.14%
Dice score: 0.0
Epoch [50/50]


100%|██████████| 1/1 [00:00<00:00,  3.77it/s, loss=0.439]


=> Saving checkpoint
Got 6086/8100 with acc 75.14%
Dice score: 0.0
